# 02. $p$-variation

**Scope.** Computing roughness of a path from samples, and reading the answer.

Relevance: $p$-variation is a **diagnostic** rather than a loss (17/08). The index floors the sampling budget (`docs/logbook/2026-08-12.md`) and decides whether the rough-path objection to pointwise losses applies to our data. $V_p$ ignores sample times, so the 13/08 uniform-sampling assumption leaves it untouched.

**Contents.** §1 definition and standard facts. §§2 to 4 the three implementations in `src/pathloss/pvar.py`, by increasing speed. §5 the one-dimensional algorithm (unimplemented). §6 timings. §7 the index estimator (unimplemented).

---

In [1]:
%matplotlib inline
import sys, pathlib, time
import numpy as np
import matplotlib.pyplot as plt

# `pathloss` comes from `pip install -e .`; this line also makes the notebook
# work without it.
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from pathloss.pvar import p_variation_brute, p_variation_exact, p_variation_pruned
from pathloss.paths import brownian_motion

rng = np.random.default_rng(0)
plt.rcParams.update({"figure.figsize": (10, 3.6), "figure.dpi": 110})

## 1. Definition and standard facts

For a sampled path $x_0, \dots, x_N$ in a metric space $(E, d)$ and $p \ge 1$,

$$V_p(x)^p \;=\; \sup \sum_k d\big(x_{n_k}, x_{n_{k-1}}\big)^p ,$$

over increasing subsequences $0 = n_0 < \cdots < n_K = N$.

### 1.1 Supremum, and the exponent

A sum over a given grid is a property of that grid; the supremum is a property of the path, since relabelling $t$ changes which partitions are available without changing the achievable sums. So $V_p$ is invariant under reparametrisation.

The exponent decides whether the supremum is trivial. For same-sign increments $a, b$,

$$|a+b|^p \;\begin{cases} = |a|^p + |b|^p, & p = 1 \\ > |a|^p + |b|^p, & p > 1 .\end{cases}$$

At $p = 1$ refining never loses, so the finest partition wins and $V_1$ is a cumulative sum, linear in $N$. At $p > 1$ coarsening can gain, so the optimum lies between and finding it is an optimisation. Butkus & Norvaiša (2018, p. 361) give the smallest example: a four-value path at $p=3$ where every local extremum scores $17/27$ against $1$ for the endpoints alone.

Ferrucci, Perrée & Lyons (2026, Remark 2.2): *"for the $p$-variation control with $p>1$ ... even computing the ordinary $p$-variation requires an optimisation over partitions."*

### 1.2 Standard facts

- $V_p$ is non-increasing in $p$, opposite monotonicity to $\|f\|_{L^p}$.
- $p = 1$ is total variation; $V_1 < \infty$ iff Riemann-Stieltjes integration against $f$ is defined.
- Brownian motion has $V_p < \infty$ a.s. exactly for $p > 2$, so $p = 2$ is critical.
- $\alpha$-Hölder $\Rightarrow V_{1/\alpha} < \infty$, since $\alpha p = 1$ makes $\sum_i (\Delta t_i)^{\alpha p}$ telescope to $T$ for every partition. Converse fails without reparametrisation (`docs/logbook/2026-08-12.md`, Reading 2).
- A path of finite $p$-variation is determined by $\lfloor p \rfloor$ signature levels, higher levels following (Lyons, 1998). Statement about describing a path as a rough path; truncation level in a learned feature map is set empirically (17/08).

### 1.3 Limitation shared by all three implementations

All three take the supremum over subsequences of the **observed grid**. True $V_p$ admits any partition of $[0,T]$, so every number here is a **lower bound** that no finite sample closes. Notebook 01 §3.1 states the same limitation for norms.

---

## 2. Implementation 1: enumeration

`p_variation_brute(x, p, dist=None, max_points=20)`

### 2.1 Definition

Every subsequence containing both endpoints, scored, largest returned. With $N+1$ points the interior has $N-1$ optional members, so $2^{N-1}$ candidates.

### 2.2 Purpose

Oracle for the other two in `tests/test_pvar.py`. It holds no argument that could be subtly wrong, only a loop over every candidate.

### 2.3 Cost, and the input limit

$2^{N-1}$ subsequences, each $O(N)$. Refuses inputs above 20 points: a test slow enough to be skipped stops being run.

### 2.4 Code

```python
for mask in range(1 << interior):        # every subset of the interior
    idx = [0] + [i + 1 for i in range(interior) if (mask >> i) & 1] + [n - 1]
    total = sum(d(a, b) ** p for a, b in zip(idx[:-1], idx[1:]))
```

The bit mask is the subset. Endpoints are prepended and appended rather than optional, the definition fixing $n_0 = 0$ and $n_K = N$.

---

## 3. Implementation 2: dynamic programming

`p_variation_exact(x, p, dist=None, return_points=False)`

### 3.1 Recursion

Ask, for each index $j$, the best score for a subsequence **ending at** $j$. Call it $D[j]$.

Such a subsequence has a last step, from some $m < j$. Fix $m$: everything before it is a subsequence ending at $m$, and must be optimal, since substituting a better prefix raises the total and leaves $d(x_m, x_j)^p$ untouched. So

$$D[0] = 0, \qquad D[j] \;=\; \max_{m<j}\big(D[m] + d(x_m,x_j)^p\big), \qquad V_p = D[N]^{1/p} .$$

Optimal substructure: prefix and last step interact only through $m$, so the exponential disappears.

### 3.2 Cost

$N$ values of $j$, each scanning up to $N$ candidates: $O(N^2)$ time, $O(N)$ memory beyond the path.

### 3.3 Code

```python
for j in range(1, n):
    incr = np.linalg.norm(x[j] - x[:j], axis=-1) ** p   # d(x_m, x_j)^p for all m < j
    cand = best[:j] + incr                              # D[m] + that
    k = int(np.argmax(cand))
    best[j], link[j] = cand[k], k
```

One vectorised row per $j$. `link` records the arg-max, so `return_points=True` recovers the maximising subsequence by walking back from $N$. Links are bookkeeping alongside the value and can drift from it, hence the test that the reported subsequence achieves the reported number.

### 3.4 Two consequences of the recursion

**Monotonicity.** $D[j+1] \ge D[j] + d(x_j,x_{j+1})^p \ge D[j]$, so $D$ is non-decreasing. §4 rests on this.

**A supplied metric costs.** With `dist=None` a row is one NumPy call; with a metric supplied it is $j$ Python calls. Asymptotics hold, the constant rises by orders of magnitude.

---

## 4. Implementation 3: pruned search

`p_variation_pruned(x, p, dist=None, return_points=False)`

A port of `p_var_backbone` from [khumarahn/p-var](https://github.com/khumarahn/p-var) (Korepanov, Lyons, Zorin-Kranich). Same recursion as §3, visiting few of the candidates.

### 4.1 Pruning threshold $\delta$

Scan $m$ downward from $j-1$, holding $\texttt{best}$, largest value so far for endpoint $j$. Candidate $m$ improves on it iff

$$D[m] + d(x_m,x_j)^p > \texttt{best}
\qquad\Longleftrightarrow\qquad
d(x_m,x_j) \;>\; \big(\texttt{best} - D[m]\big)^{1/p} \;=:\; \delta .$$

$\delta$ is the distance $x_m$ must be from $x_j$ to be worth visiting. $D$ is non-decreasing (§3.4), so $D[m]$ falls as $m$ falls and $\delta$ **rises** as the scan runs back: an earlier index carries less accumulated score, so its single step must make up the difference.

### 4.2 Block bound

Partition the indices into dyadic blocks. For a block $B$ with centre $c$ and radius $R_B = \max_{m' \in B} d(x_{m'}, x_c)$, the triangle inequality gives, for every $m' \in B$,

$$d(x_{m'}, x_j) \;\le\; d(x_{m'}, x_c) + d(x_c, x_j) \;\le\; R_B + d(x_c, x_j).$$

If that bound is at most $\delta$, no member of $B$ passes the §4.1 test, and all $|B|$ are skipped on one distance evaluation. Searching by postcode rather than by door.

Only the triangle inequality and symmetry are used, so `dist` may be any metric. The project needs that: rough-path $p$-variation is measured in the homogeneous norm on signatures rather than on values, and the same routine computes it.

### 4.3 Dyadic blocks

Each index lies in $\log N$ dyadic blocks, nested across levels. So radii accumulate online as $j$ advances at $\log N$ updates per step, and the scan tries the largest block first, shrinking until the bound bites. `radius` packs all levels into one flat array of length $N-1$ at position `(s >> k) + (j >> k)`, with a guard for the truncated final block.

Here "dyadic" is the shape of a search tree, changing no answer, unlike a restriction on which partitions are considered.

### 4.4 Cost, and the worst case on a straight line

About $O(N\log N)$ on a path that changes direction. $O(N^2)$ on a monotone path in $\mathbb{R}$, where

$$d(x_{m'}, x_c) + d(x_c, x_j) = d(x_{m'}, x_j)$$

**exactly**: the bound has no slack, so nothing is excluded. Pruning needs a strict triangle inequality, and a straight line gives equality. A wandering path returns near where it has been, so going further back adds little distance while $\delta$ keeps rising, and whole blocks fall away.

### 4.5 Lazy update of $\delta$

A $\delta$ computed at a later $m$ is smaller than the current one, hence conservative, so the cheap comparison runs first and the $p$-th root is taken only when it fails. Correctness survives because too small a $\delta$ admits candidates that the explicit comparison then rejects.

---

## 5. The one-dimensional algorithm (not implemented)

Butkus & Norvaiša (2018), R package `pvar`. A third strategy: rather than searching more cleverly, **make the problem smaller**.

**Step 1, corners.** A maximising partition is always a subpartition of the minimal monotonicity partition (their Theorem 1), so every point other than a local extremum can be deleted outright. One linear pass, removing most of smooth input.

**Step 2, redundant corners.** Some turning points are droppable too. Over a window $S_k, \dots, S_{k+m}$, if

$$\sum_{i=k+1}^{k+m} |S_i - S_{i-1}|^p \;<\; |S_{k+m} - S_k|^p ,$$

then hopping straight across beats the interior, so some inner point is redundant (their Corollary 4), and their Lemmas 4 and 5 upgrade "some" to "all". Sweep $m = 3, 5, 7, \dots$, deleting, then merge the surviving pieces pairwise.

**Obstruction for us.** Local extrema, monotone runs and alternating signs all require an ordered line, and a path in $\mathbb{R}^d$ has no local maxima. This is what the p-var README means by "it does not work for example in $\mathbb{R}^2$. Here we rectify this."

**Value to us.** Its best case is §4's worst case: smooth or monotone data is annihilated by step 1 and defeats the block bound of §4.2. Should real data prove smooth (13/08 starts from univariate real series), the R package is the faster route, and `FindCorners` is a ten-line preprocessing step available for $d=1$ either way.

---

## 6. Cost, measured

§3 and §4 return the same number, asserted in the cell, so the comparison is about time alone. Read the growth rate rather than the absolute times: $O(N^2)$ against roughly $O(N\log N)$, so the ratio should widen with $N$.

**Caveat.** `p_variation_exact` is one NumPy call per row, `p_variation_pruned` a Python loop. The constant favours the first heavily, putting the crossover at much larger $N$ than the asymptotics suggest, so these timings measure the implementations rather than the algorithms.

In [2]:
sizes = [2**k + 1 for k in range(7, 12)]
p = 2.5
_, W = brownian_motion(n=sizes[-1], T=1.0, d=1, rng=3)

print(f"{'n':>7} {'V_p':>10} {'t_exact':>10} {'t_pruned':>10} {'ratio':>8}")
for n in sizes:
    w = W[:n]
    t0 = time.perf_counter(); ve = p_variation_exact(w, p); t1 = time.perf_counter()
    vp = p_variation_pruned(w, p);                          t2 = time.perf_counter()
    assert abs(ve - vp) < 1e-9 * max(1.0, ve), "implementations disagree"
    print(f"{n:>7} {ve:>10.4f} {t1-t0:>10.4f} {t2-t1:>10.4f} {(t1-t0)/(t2-t1):>8.2f}")

      n        V_p    t_exact   t_pruned    ratio
    129     0.3519     0.0015     0.0083     0.18
    257     0.5523     0.0028     0.0200     0.14
    513     0.9170     0.0048     0.0446     0.11
   1025     1.4401     0.0135     0.0991     0.14
   2049     2.1993     0.0414     0.2211     0.19


## 7. Estimating the index (not implemented)

Every function above takes $p$ as an argument. The project needs

$$p^\ast = \inf\{p : V_p < \infty\},$$

which no single evaluation supplies, $V_p$ on $N$ points being finite for every $p$. The index shows in how the sum **diverges under refinement**.

### 7.1 Estimator

With increments over windows of width $2^{-k}$ scaling as $2^{-k/p^\ast}$, summing $2^k$ of them,

$$\sum_{i=1}^{2^k} |\Delta_i x|^p \;\approx\; 2^{k}\cdot 2^{-kp/p^\ast} \;=\; 2^{\,k(1 - p/p^\ast)} .$$

The exponent is positive for $p < p^\ast$, negative for $p > p^\ast$, zero at $p = p^\ast$. So regress $\log_2$ of the level-$k$ sum on $k$: the slope is $1 - p/p^\ast$, and the $p$ at which it crosses zero is the index.

Same trichotomy as §1.2's Hölder computation, in dyadic notation. There $\sum(\Delta t)^{q} = T h^{q-1}$ blows up for $q<1$, is constant at $q=1$, vanishes for $q>1$. The estimator reports which case holds at each $p$, and the crossover.

### 7.2 Two known failure modes

**Observation noise.** Additive noise has infinite variation, so it dominates at fine scales and drives the estimate upward: the microstructure problem from realised-volatility estimation. The answer therefore depends on the range of scales fitted, which must be reported with it.

**Downward bias.** The supremum over the observed grid understates the true supremum, so every $V_p$ here is a lower bound and the index inherits the bias.

### 7.3 Priority

$p^\ast$ floors the sampling budget at $N \gtrsim \epsilon^{-p^\ast}$ (`docs/logbook/2026-08-12.md`; a floor rather than a rate, finite $p$-variation bounding the Hölder exponent above without fixing it) and decides whether the rough-path objection to pointwise losses applies to our data. It is also the first quantity in the project about the data rather than about our own constructions.

---

## References

Keyed to `papers/references.bib`.

- ★ **Ferrucci, Perrée & Lyons (2026)**, arXiv:2607.26281, Remark 2.2: why $V_p$ for $p>1$ is an optimisation over partitions. §1.1.
- **Korepanov, Lyons & Zorin-Kranich**, [p-var](https://github.com/khumarahn/p-var): the pruned algorithm, valid in any metric space. §4.
- **Butkus & Norvaiša (2018)**, *Computation of $p$-variation*, Lith. Math. J. 58(4), with R package `pvar`. §1.1, §5.
- **Daoudi & Junca (2024)**, *Efficient algorithms computing $p$-variation*, preprint. §5.
- **Lyons (1998)**, *Differential equations driven by rough signals*, Rev. Mat. Iberoamericana 14(2): $\lfloor p \rfloor$ levels suffice. §1.2.